# 銷售分析｜營收成長與訂單趨勢

商業問題：  
月度營收趨勢與淡旺季特徵分析  
分析方法：  
- 按年月彙總營收與訂單數
- 計算`月增率（ MoM ）`
- 比較各月份營收變化，觀察淡旺季特徵

In [ ]:
WITH monthly_sales AS (
     SELECT CONVERT(varchar(7),Order_Date,120) AS [年-月],
            COUNT(DISTINCT Order_ID) AS [訂單數],
            ROUND(SUM(Total_Amount),2) AS [本月營收]
       FROM india_ecom.dbo.sales
      GROUP BY CONVERT(varchar(7),Order_Date,120))

SELECT [年-月],
       [訂單數],
       [本月營收],
       LAG([本月營收],1) OVER(ORDER BY [年-月]) AS [上月營收],
      ROUND(([本月營收]-LAG([本月營收],1) OVER(ORDER BY [年-月]))*100.0/LAG([本月營收],1) OVER(ORDER BY [年-月]),2) AS [MOM]
  FROM monthly_sales
 ORDER BY [年-月];

(25 個資料列受到影響)

年-月     | 訂單數   | 本月營收         | 上月營收         | MOM   
--------+-------+--------------+--------------+-------
2024-06 | 10008 | 240781697.54 | NULL         | NULL  
2024-07 | 10216 | 240966259.7  | 240781697.54 | 0.08  
2024-08 | 10158 | 240194624.64 | 240966259.7  | -0.32 
2024-09 | 9704  | 229331306.32 | 240194624.64 | -4.52 
2024-10 | 10034 | 239971596.69 | 229331306.32 | 4.64  
2024-11 | 9865  | 234557799.83 | 239971596.69 | -2.26 
2024-12 | 10156 | 241951018.11 | 234557799.83 | 3.15  
2025-01 | 10243 | 241229155.52 | 241951018.11 | -0.3  
2025-02 | 9211  | 215193135.77 | 241229155.52 | -10.79
2025-03 | 10296 | 242631110.45 | 215193135.77 | 12.75 
2025-04 | 9961  | 241486810.88 | 242631110.45 | -0.47 
2025-05 | 10299 | 241196492.63 | 241486810.88 | -0.12 
2025-06 | 9919  | 232215999.76 | 241196492.63 | -3.72 
2025-07 | 10131 | 238134009.62 | 232215999.76 | 2.55  
2025-08 | 10201 | 246113996.33 | 238134009.62 | 3.35  
2025-09 | 9819  | 228021991.01 | 246113996.33 | -7

分析結果：  
整體營收大致在 2.15~2.48 億間震盪，無明顯成長趨勢，屬平穩波動型。最大跌幅出現在 2025 年 2 月（ -10.79% ），最大漲幅為 2025 年 3 月（ +12.75% ），二月天數較少，可能是當月營收下滑的因素之一，而三月營收出現明顯回升，可進一步與行銷檔期進行比對，探討灑紅節（ Holi ）對訂單與營收的影響。

商業問題：  
地區業績貢獻與營收集中度分析    
分析方法：  
- 彙總各 State 的總營收並排序

In [1]:
SELECT TOP(10) State,
       round(SUM(Total_Amount),2) AS [總金額]
  FROM india_ecom.dbo.sales
 GROUP BY State
 ORDER BY [總金額] DESC;

(10 個資料列受到影響)

State       | 總金額         
------------+-------------
UP          | 768351008.57
Rajasthan   | 757286462   
Haryana     | 755215239.23
Maharashtra | 605830687.35
Punjab      | 599225410.94
Gujarat     | 599177742.04
Delhi       | 474469227.07
Tamil Nadu  | 470515707.09
West Bengal | 456319987.07
Karnataka   | 444301789.01
(10 個資料列)

總執行時間: 00:00:02.561

分析結果：  
查詢結果列出營收前 10 大的邦（本資料集僅涵蓋 10 個邦，即全部邦別）。營收最高為 UP （約 7.68 億），其次是 Rajasthan （約 7.57 億）與 Haryana （約 7.55 億），三者營收相近、地理位置均為北印，且明顯領先其他邦； Maharashtra、Punjab、Gujarat 居中， Delhi、Tamil Nadu、West Bengal、Karnataka 則相對較低。整體來看營收分佈集中在北印度的幾個邦別，前三名合計已占相當比重，可作為區域資源分配或行銷投放優先順序的參考依據。

商業問題：  
取消訂單與退貨對實質營收的影響  
分析方法：  
- 依 `Order_Status` 彙總訂單數與營收
- 計算各訂單狀態的訂單占比
- 比較取消與退貨對營收的影響

In [2]:
WITH  cancel AS (SELECT Order_Status AS [訂單狀態],
                        COUNT(Order_Status) AS [訂單數],
                        round(SUM(Total_Amount),2) AS [總金額]
                   FROM india_ecom.dbo.sales
                  GROUP BY Order_Status)
SELECT [訂單狀態],
       [訂單數],
       [總金額],
       [訂單數]*100/SUM([訂單數]) OVER() AS [訂單佔比]
  FROM cancel
 ORDER BY [訂單佔比] DESC;

(5 個資料列受到影響)

訂單狀態       | 訂單數    | 總金額           | 訂單佔比
-----------+--------+---------------+-----
Delivered  | 200139 | 4741566359.35 | 80  
Cancelled  | 12507  | 302140522.18  | 5   
Processing | 12402  | 295460811.49  | 4   
Returned   | 12493  | 300404736.67  | 4   
Shipped    | 12459  | 291120830.69  | 4   
(5 個資料列)

總執行時間: 00:00:02.498

分析結果：  
統計各訂單狀態的訂單數、總營收與訂單佔比的結果，顯示：「已送達（ Delivered ）」約佔 80% 訂單、貢獻約 47.4 億營收；其餘「取消（ Cancelled ）」、「處理中（ Processing ）」、「已退貨（ Returned ）」及「已出貨（ Shipped ）」各約佔 4-5% ，整體來看訂單完成率良好。但取消與退貨合計約占一成訂單，仍值得進一步追蹤其成因（如商品品質、物流延遲等）以降低損耗。

商業問題：  
付款方式與物流成本結構分析  
分析方法：  
- 依 `Payment_Mode` 彙總平均消費金額與平均運費
- 計算物流成本占比
- 比較各付款方式的成本結構

In [6]:
SELECT Payment_Mode AS [付款方式],
       round(AVG(Order_Value),2) AS [平均消費金額],
       round(AVG(Shipping_Cost),2) AS [平均運費],
       round((SUM(CAST(Shipping_Cost AS FLOAT))/NULLIF(SUM(Order_Value), 0))*100,2) AS [物流成本佔比]
  FROM india_ecom.dbo.sales
 GROUP BY Payment_Mode
 ORDER BY [平均消費金額] DESC;

(4 個資料列受到影響)

付款方式        | 平均消費金額   | 平均運費 | 物流成本佔比
------------+----------+------+-------
Credit Card | 26812.27 | 4.4  | 0.02  
UPI         | 24980.1  | 4.71 | 0.02  
COD         | 22665.46 | 5.16 | 0.02  
Debit Card  | 22123.76 | 5.27 | 0.02  
(4 個資料列)

總執行時間: 00:00:00.794

分析結果：  
透過比較各付款方式的平均消費金額、平均運費與物流成本佔比，觀察運費對付款方式的影響。結果顯示 Credit Card 的 AOV 最高（約 26,812 ）， COD 與 Debit Card 則偏低（約 22,000 多）；平均運費則差異不大（約 4.4~5.3 ），物流成本佔訂單金額的比例都僅約 0.02% 。顯示物流成本占訂單金額比例很低，各付款方式間的差異主要來自消費金額而非物流成本。